# F0 · Setup de la API (vlrggapi) en Google ColabMonta Google Drive, clona el repo de [vlrggapi](https://github.com/axsddlr/vlrggapi),instala dependencias, arranca el servidor local y hace un *smoke test*.

## 0. Montar Google DriveLos datos se persisten en Drive para sobrevivir desconexiones del runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Configurar rutas

In [ ]:
import sys
from pathlib import Path

# Ajusta esta ruta a donde tengas el repo dentro de tu Drive.
DRIVE_BASE = "/content/drive/MyDrive/Analisis de datos/Proyecto final"
# Si prefieres el disco local de Colab (más rápido, efímero), descomenta:
# DRIVE_BASE = "/content/Proyecto final"

REPO_DIR = Path(DRIVE_BASE)
DATA_DIR = REPO_DIR / "data"
SRC_DIR = REPO_DIR / "src"
API_DIR = REPO_DIR / "vlrggapi"
BASE_URL = "http://127.0.0.1:3001"

for d in (REPO_DIR, DATA_DIR, SRC_DIR):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SRC_DIR))
print("REPO_DIR:", REPO_DIR)
print("DATA_DIR:", DATA_DIR)


## 2. Clonar el repo del proyecto (opcional) y vlrggapiEl repo del proyecto normalmente ya lo subiste a Drive. Aquí solo clonamos**vlrggapi** (la API local).

In [ ]:
# Clona vlrggapi (la instancia hosteada está caída). Si ya existe, no pasa nada.
!git clone https://github.com/axsddlr/vlrggapi.git "{API_DIR}" 2>/dev/null || echo "vlrggapi ya clonado"


## 3. Instalar dependencias

In [ ]:
# Dependencias del proyecto (pandas, pyarrow, requests, tqdm, ...).
!pip install -q -r "{REPO_DIR}/requirements.txt"
# Dependencias de vlrggapi (uv no está en Colab; usamos pip directamente).
!pip install -q fastapi "uvicorn[standard]" httpx selectolax cachetools

# Clona vlrggapi (la instancia hosteada está caída). Si ya existe, no pasa nada.
!git clone https://github.com/axsddlr/vlrggapi.git "{API_DIR}" 2>/dev/null || echo "vlrggapi ya clonado"


## 4. Arrancar vlrggapi en segundo plano

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

from scraper import VLRClient, start_server, wait_for_api

# Arranca el servidor si no responde (idempotente entre notebooks/runtime).
if not wait_for_api(BASE_URL, timeout=5):
    proc = start_server(API_DIR, port=3001, log_file=DATA_DIR / "_vlrggapi.log")
    wait_for_api(BASE_URL, timeout=240)

client = VLRClient(base_url=BASE_URL, data_dir=DATA_DIR)
print("API lista:", client.version())


## 5. Smoke test`/version` y `/v2/health` son tier *cheap* (200/min): no gastan presupuesto.Ambos deben responder sin tocar vlr.gg.

In [ ]:
from scraper import VLRClient

client = VLRClient(base_url=BASE_URL, data_dir=DATA_DIR)
print("version:", client.version())
print("health :", client.health())


## 6. Ejemplo de llamada (comentado)Descomenta para probar una llamada real. Recuerda: `/v2/match?q=results` estier *expensive* (20/min).

In [ ]:
# Ejemplo: una página de resultados recientes.
# data = client.match_results(num_pages=1)
# print("partidos:", len(data.get("segments", [])), "| meta:", data.get("meta"))
